# VQ-VAE-2 Latent Feature Extraction

Extract and visualize latent representations from all 3 hierarchical levels of the VQ-VAE-2 encoder.

The new model architecture includes a `content_proj` layer (Conv3d) that projects content-only channels
back to `hidden_channels` after the level-0 encoder, so deeper encoders receive a full-channel spatial map.
This notebook correctly accounts for that by:
- Building the model with `content_size` and `style_size` (required to reconstruct `content_proj`)
- Using `pool_only=True` to get pooled `(B, C)` vectors per level (memory-efficient)
- Separately extracting **content** vs **style** channel subsets at level 0 using the Gumbel mask
- Visualising T1 vs T2 separation and diagnosis clustering at each hierarchical level

In [ ]:
import sys
import os

# Add project root to sys.path (this notebook lives in eval/)
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import json

# Import local modules
import models.vqvae as vqvae
import utils.utils as utils
from utils.utils import load_data, CreateBrainMaskd, ApplyBrainMaskd

# ============================================================
# CONFIGURATION
# ============================================================
# Use vqvae_best.pt for the best checkpoint (by rolling avg total loss),
# or vqvae_model.pt for the latest checkpoint.
CHECKPOINT_PATH = "/home/ng24/projects/multiview-crl/results/ADNI_registered/vqvae-normal-logging-4/vqvae_best.pt"
DATA_DIR = "/data/natalia/ADNI_registered"
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
NUM_SAMPLES = 100  # number of subjects to process
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ADNI 2-view split (must match training)
CONTENT_SIZE = 256  # dimensions treated as content (shared between T1/T2)
STYLE_SIZE = 256  # dimensions treated as style (view-specific)

# Resampling: "original" (1 mm, ~182x218x182) or "2mm" (91x109x91)
RESAMPLE_MODE = "original"
# ============================================================

print(f"Using device: {DEVICE}")

## 1. Load checkpoint & build model

In [ ]:
# Load settings saved alongside the checkpoint
settings_path = os.path.join(os.path.dirname(CHECKPOINT_PATH), "settings.json")
with open(settings_path, "r") as f:
    settings = json.load(f)

print("Model settings:")
for k in [
    "vqvae_hidden_channels",
    "vqvae_res_channels",
    "vqvae_nb_res_layers",
    "vqvae_nb_levels",
    "vqvae_embed_dim",
    "vqvae_nb_entries",
    "vqvae_scaling_rates",
]:
    print(f"  {k}: {settings.get(k, 'N/A')}")

# Override CONTENT_SIZE / STYLE_SIZE from saved settings.
# settings.json stores content_indices (list[list[int]]) and style_indices (list[int])
# written by update_args() at training time — use them so the VQVAE is rebuilt
# with exactly the same content/style ratio that was used during training.
# Falling back to the hardcoded values only when the keys are absent (old checkpoints).
if "content_indices" in settings and "style_indices" in settings:
    CONTENT_SIZE = len(settings["content_indices"][0])
    STYLE_SIZE = len(settings["style_indices"])
    print(f"\n  content_size : {CONTENT_SIZE}  (from settings.json['content_indices'])")
    print(f"  style_size   : {STYLE_SIZE}  (from settings.json['style_indices'])")
else:
    print(f"\n  ⚠ content_indices / style_indices not found in settings.json.")
    print(f"    Using hardcoded CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}.")
    print(f"    If these don't match the training run, content_channels will be wrong.")

In [ ]:
# Build model — must match the exact architecture used during training.
# inject_style_to_decoder controls whether decoders[0] has a final_conv branch;
# if it differs from the checkpoint the decoder weights will silently not load.
inject_style = settings.get("inject_style_to_decoder", False)

vqvae_model = vqvae.VQVAE(
    in_channels=1,
    hidden_channels=settings["vqvae_hidden_channels"],
    res_channels=settings["vqvae_res_channels"],
    nb_res_layers=settings.get("vqvae_nb_res_layers", 2),
    nb_levels=settings["vqvae_nb_levels"],
    embed_dim=settings["vqvae_embed_dim"],
    nb_entries=settings["vqvae_nb_entries"],
    scaling_rates=settings["vqvae_scaling_rates"],
    content_size=CONTENT_SIZE,
    style_size=STYLE_SIZE,
    inject_style_to_decoder=inject_style,
)

hidden_channels = settings["vqvae_hidden_channels"]
content_channels = vqvae_model.content_channels  # actual channel count (may differ due to rounding)
print(f"hidden_channels       : {hidden_channels}")
print(f"content_channels      : {content_channels}")
print(f"inject_style_to_decoder: {inject_style}")

# Wrap in DataParallel to match the saved checkpoint structure
vqvae_model = torch.nn.DataParallel(vqvae_model, device_ids=[0])

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = checkpoint["encoders"]

# ── Key remapping ─────────────────────────────────────────────────────────────
# Checkpoints can come from three different wrapper configurations:
#
#   1. Bare VQVAE in DataParallel           → keys: "module.*"
#   2. MoCoEncoder wrapping DataParallel    → keys: "online.module.*",
#                                                    "momentum_encoders.*",
#                                                    "momentum_content_proj.*",
#                                                    "queue_N"
#
# For case 2 we strip the "online." prefix and discard MoCo-only keys.
# We also apply the legacy ".blocks." → ".stack." rename for old checkpoints.

new_state_dict = {}
is_moco_checkpoint = any(k.startswith("online.") for k in state_dict)
if is_moco_checkpoint:
    print("Detected MoCoEncoder checkpoint — stripping 'online.' prefix.")

for k, v in state_dict.items():
    k = k.replace(".blocks.", ".stack.")  # legacy rename
    if k.startswith("online."):
        # Strip the MoCoEncoder wrapper prefix: "online.module.*" → "module.*"
        new_state_dict[k[len("online.") :]] = v
    elif k.startswith(("momentum_encoders.", "momentum_content_proj.", "queue_")):
        pass  # MoCo-only state — not part of the bare VQVAE, discard
    else:
        new_state_dict[k] = v

missing, unexpected = vqvae_model.load_state_dict(new_state_dict, strict=False)

# Filter out num_batches_tracked mismatches — these are benign initialisation
# differences in BatchNorm and don't affect model behaviour.
missing_real = [k for k in missing if "num_batches_tracked" not in k]
unexpected_real = [k for k in unexpected if "num_batches_tracked" not in k]

if missing_real:
    print(f"⚠ Missing keys ({len(missing_real)}): {missing_real[:6]}{'...' if len(missing_real) > 6 else ''}")
if unexpected_real:
    print(
        f"⚠ Unexpected keys ({len(unexpected_real)}): {unexpected_real[:6]}{'...' if len(unexpected_real) > 6 else ''}"
    )
if not missing_real and not unexpected_real:
    print("✓ Checkpoint loaded cleanly")
elif not missing_real:
    print("✓ Checkpoint loaded (unexpected keys are harmless extras in the file)")

vqvae_model.to(DEVICE)
vqvae_model.eval()
print(f"✓ Model on {DEVICE}, step {checkpoint['step']}")
print(f"  Total params: {sum(p.numel() for p in vqvae_model.parameters()):,}")

## 2. Load dataset & transforms

In [ ]:
df = pd.read_csv(CSV_PATH)
label_values = sorted(df["Group"].unique())
label_map = {v: i for i, v in enumerate(label_values)}
label_names = {i: v for v, i in label_map.items()}
print(f"Labels: {label_map}")

items, missing = load_data(df, DATA_DIR, label_map)
print(f"Loaded {len(items)} samples, {len(missing)} missing files")
items = items[:NUM_SAMPLES]
print(f"Using first {len(items)} subjects")

In [ ]:
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd
from monai.transforms import ResizeWithPadOrCropd, NormalizeIntensityd, ToTensord

# Map RESAMPLE_MODE to the spacing value used by utils.utils.transforms()
if RESAMPLE_MODE == "2mm":
    spacing = 2.0
elif RESAMPLE_MODE == "original":
    spacing = 1.0
else:
    raise ValueError(f"Unknown RESAMPLE_MODE: {RESAMPLE_MODE}")

# Re-use the project's standard transform builder, but drop the label key
# (this notebook only works with image pairs, no label tensor needed)
base_keys = ["image_t1", "image_t2"]
mask_keys = ["mask_t1", "mask_t2"]
all_keys = base_keys + mask_keys

if spacing == 1.0:
    spatial_size = (182, 218, 182)
else:
    spatial_size = (91, 109, 91)

pre = [
    LoadImaged(keys=base_keys),
    EnsureChannelFirstd(keys=base_keys, channel_dim="no_channel"),
    CreateBrainMaskd(keys=base_keys, mask_keys=mask_keys),
    Orientationd(keys=all_keys, axcodes="RAS"),
]

if spacing != 1.0:
    pre += [
        Spacingd(keys=base_keys, pixdim=(spacing, spacing, spacing), mode="bilinear"),
        Spacingd(keys=mask_keys, pixdim=(spacing, spacing, spacing), mode="nearest"),
    ]

val_transforms = Compose(
    pre
    + [
        ResizeWithPadOrCropd(keys=all_keys, spatial_size=spatial_size),
        NormalizeIntensityd(keys=base_keys, nonzero=True, channel_wise=True),
        ApplyBrainMaskd(keys=base_keys, mask_keys=mask_keys, threshold=0.5),
        ToTensord(keys=base_keys),
    ]
)
print(f"Transforms ready — spacing={spacing}mm, spatial_size={spatial_size}")

## 3. Feature extraction

We call `vqvae_model(img, pool_only=True, return_recon=False)` which returns:
- `encoder_features`: list of `(B, C)` pooled vectors, one per level
  - **Level 0**: `content_channels` dims (the content-masked, projected feature)
  - **Level 1+**: `hidden_channels` dims (full channel map pooled)
- `estimated_content_indices`: the Gumbel-selected channel indices at level 0

For visualisation we also split level-0 pooled features into **content** and **style** subsets.

In [ ]:
nb_levels = settings["vqvae_nb_levels"]

# Storage
all_features = {f"level_{i}": [] for i in range(nb_levels)}
all_labels = []
all_modalities = []
all_subjects = []

# We'll also collect the raw level-0 FULL pooled features (hidden_channels dims)
# by running the encoder manually once, so we can split content vs style.
all_full_level0 = []  # (B, hidden_channels)
all_content_indices = None  # shared across samples (mask is computed per-forward from mean logits)

print(f"Extracting latent features from {len(items)} subjects (T1 + T2 each) ...")

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            _, _, enc_features, est_content_idx, _, _ = vqvae_model(
                img,
                return_recon=False,
                pool_only=True,
            )

        # enc_features is a list of (1, C) pooled tensors
        for level_idx, pooled in enumerate(enc_features):
            all_features[f"level_{level_idx}"].append(pooled.squeeze(0).cpu().float().numpy())  # (C,)

        # For level-0: also grab the FULL hidden_channels pooled features
        # by running just encoder[0] directly (before content projection path)
        with torch.no_grad():
            spatial_l0 = vqvae_module.encoders[0](img)  # (1, hidden_channels, D, H, W)
            full_pool = spatial_l0.mean(dim=[2, 3, 4])  # (1, hidden_channels)
        all_full_level0.append(full_pool.squeeze(0).cpu().float().numpy())

        # Record the content indices from the last forward pass
        if all_content_indices is None and est_content_idx is not None:
            all_content_indices = est_content_idx[0]  # list of ints

        all_labels.append(item["label"])
        all_modalities.append(modality)
        all_subjects.append(item.get("subject", idx))

# Convert to numpy
for key in all_features:
    all_features[key] = np.array(all_features[key])
all_full_level0 = np.array(all_full_level0)
all_labels = np.array(all_labels)
all_modalities = np.array(all_modalities)

print("\n✓ Feature extraction complete!")
for key, val in all_features.items():
    print(f"  {key}: {val.shape}")
print(f"  full level-0 (hidden_channels): {all_full_level0.shape}")

# Derive content / style subsets from the full level-0 features
all_style_indices = [i for i in range(hidden_channels) if i not in set(all_content_indices or [])]
print(f"  content indices ({len(all_content_indices)} ch): {all_content_indices[:8]}...")
print(f"  style indices   ({len(all_style_indices)} ch):   {all_style_indices[:8]}...")

## 4. Feature statistics

In [ ]:
print("=" * 65)
print("Feature Statistics by Level (pooled, before quantization)")
print("=" * 65)

t1_mask = all_modalities == "T1"
t2_mask = all_modalities == "T2"

for level_idx in range(nb_levels):
    features = all_features[f"level_{level_idx}"]
    t1_f = features[t1_mask]
    t2_f = features[t2_mask]
    paired_dist = np.linalg.norm(t1_f - t2_f, axis=1)
    print(f"\nLevel {level_idx}  (shape {features.shape}):")
    print(
        f"  mean={features.mean():.4f}  std={features.std():.4f}  "
        f"min={features.min():.4f}  max={features.max():.4f}"
    )
    print(f"  T1-T2 paired ℓ2 distance — mean: {paired_dist.mean():.4f}  std: {paired_dist.std():.4f}")

# Extra breakdown for level 0: content vs style channels
if all_content_indices is not None:
    print("\n--- Level 0 channel breakdown (full hidden_channels pool) ---")
    content_f = all_full_level0[:, all_content_indices]
    style_f = all_full_level0[:, all_style_indices] if all_style_indices else None
    ct1 = content_f[t1_mask]
    ct2 = content_f[t2_mask]
    print(
        f"  Content  ({len(all_content_indices)} ch)  — T1-T2 paired dist: {np.linalg.norm(ct1-ct2,axis=1).mean():.4f}"
    )
    if style_f is not None and len(style_f):
        st1 = style_f[t1_mask]
        st2 = style_f[t2_mask]
        print(
            f"  Style    ({len(all_style_indices)} ch)  — T1-T2 paired dist: {np.linalg.norm(st1-st2,axis=1).mean():.4f}"
        )
    print("  (Content dist should be smaller than style dist if disentanglement is working)")

## 5. PCA visualisation

In [ ]:
colors_by_label = plt.cm.tab10(np.linspace(0, 1, max(len(label_map), 3)))

fig, axes = plt.subplots(2, nb_levels, figsize=(5 * nb_levels, 9))
if nb_levels == 1:
    axes = axes[:, np.newaxis]  # keep 2-D indexing

for level_idx in range(nb_levels):
    # For level 0, show only the content channels — these are what the contrastive
    # loss acts on.  Plotting all hidden_channels at level 0 mixes in style channels,
    # which encode T1/T2 contrast differences and will dominate the first PC,
    # making the plot cluster by modality regardless of disentanglement quality.
    # The full content+style breakdown for level 0 is in Section 7.
    if level_idx == 0 and all_content_indices is not None:
        features = all_full_level0[:, all_content_indices]
        level_label = f"Level 0 — content ({len(all_content_indices)} ch)"
    else:
        features = all_features[f"level_{level_idx}"]
        level_label = f"Level {level_idx}"

    pca = PCA(n_components=2)
    f2d = pca.fit_transform(features)
    ev = pca.explained_variance_ratio_

    # Row 0 — colour by diagnosis
    ax = axes[0, level_idx]
    for lab_idx, lab_name in label_names.items():
        m = all_labels == lab_idx
        ax.scatter(
            f2d[m, 0],
            f2d[m, 1],
            c=[colors_by_label[lab_idx]],
            label=lab_name,
            alpha=0.65,
            s=18,
        )
    ax.set_title(f"{level_label} — Diagnosis\n({ev.sum()*100:.1f}% var)")
    ax.legend(fontsize=8)
    ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

    # Row 1 — colour by modality
    ax = axes[1, level_idx]
    for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
        m = all_modalities == mod
        ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
    ax.set_title(f"{level_label} — Modality (T1 vs T2)")
    ax.legend(fontsize=8)
    ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

plt.suptitle(
    "VQ-VAE-2 Encoder Features — PCA\n"
    "(Level 0: content channels only; Levels 1+: full hidden_channels; all pooled pre-quantization)",
    y=1.02,
    fontsize=12,
)
plt.tight_layout()
plt.savefig("latent_features_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved latent_features_pca.png")

## 6. t-SNE visualisation

In [ ]:
fig, axes = plt.subplots(2, nb_levels, figsize=(5 * nb_levels, 9))
if nb_levels == 1:
    axes = axes[:, np.newaxis]

for level_idx in range(nb_levels):
    # Apply the same content-only filtering as the PCA section for level 0.
    # Without this, style channels (which encode T1/T2 contrast differences)
    # dominate the embedding and create spurious modality clusters.
    if level_idx == 0 and all_content_indices is not None:
        features = all_full_level0[:, all_content_indices]
        level_label = f"Level 0 — content ({len(all_content_indices)} ch)"
    else:
        features = all_features[f"level_{level_idx}"]
        level_label = f"Level {level_idx}"

    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(features) - 1))
    f2d = tsne.fit_transform(features)

    ax = axes[0, level_idx]
    for lab_idx, lab_name in label_names.items():
        m = all_labels == lab_idx
        ax.scatter(
            f2d[m, 0],
            f2d[m, 1],
            c=[colors_by_label[lab_idx]],
            label=lab_name,
            alpha=0.65,
            s=18,
        )
    ax.set_title(f"{level_label} — Diagnosis")
    ax.legend(fontsize=8)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

    ax = axes[1, level_idx]
    for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
        m = all_modalities == mod
        ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
    ax.set_title(f"{level_label} — Modality (T1 vs T2)")
    ax.legend(fontsize=8)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

plt.suptitle(
    "VQ-VAE-2 Encoder Features — t-SNE\n"
    "(Level 0: content channels only; Levels 1+: full hidden_channels; all pooled pre-quantization)",
    y=1.02,
    fontsize=13,
)
plt.tight_layout()
plt.savefig("latent_features_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved latent_features_tsne.png")

## 7. Content vs Style channel visualisation (level 0)

The Gumbel mask at level 0 selects `content_channels` out of `hidden_channels`.
If the model has learned to disentangle content (shared across T1/T2) from style
(view-specific), the **content** PCA should show T1 and T2 mixed together, while
the **style** PCA should separate them clearly.

In [ ]:
if all_content_indices is not None and len(all_style_indices) > 0:
    content_feats = all_full_level0[:, all_content_indices]
    style_feats = all_full_level0[:, all_style_indices]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    for row, (feats, title_sfx) in enumerate(
        [
            (content_feats, f"Content ({len(all_content_indices)} ch)"),
            (style_feats, f"Style   ({len(all_style_indices)} ch)"),
        ]
    ):
        pca = PCA(n_components=2)
        f2d = pca.fit_transform(feats)
        ev = pca.explained_variance_ratio_

        # Col 0 — by diagnosis
        ax = axes[row, 0]
        for lab_idx, lab_name in label_names.items():
            m = all_labels == lab_idx
            ax.scatter(
                f2d[m, 0],
                f2d[m, 1],
                c=[colors_by_label[lab_idx]],
                label=lab_name,
                alpha=0.65,
                s=18,
            )
        ax.set_title(f"Level-0 {title_sfx}\nBy Diagnosis ({ev.sum()*100:.1f}% var)")
        ax.legend(fontsize=8)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

        # Col 1 — by modality
        ax = axes[row, 1]
        for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
            m = all_modalities == mod
            ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
        ax.set_title(f"Level-0 {title_sfx}\nBy Modality")
        ax.legend(fontsize=8)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

    plt.suptitle("Level-0 Content vs Style Channel Subsets — PCA", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig("latent_content_vs_style_pca.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Saved latent_content_vs_style_pca.png")
else:
    print("Skipped — content_proj not active or all channels are content.")

## 8. Reconstruction visualisation

Compare original slices with their VQ-VAE-2 reconstructions.

## 9. Diagnostic label prediction

For each feature set (all encoder levels + level-0 content/style subsets) we train a
**logistic regression** and a **random forest** classifier on T1 features only (so modality
doesn't leak into the score), using stratified 5-fold cross-validation.

This tells us:
- **Which hierarchical level** encodes the most diagnostically-relevant information
- **Whether content or style channels** at level 0 carry the diagnostic signal
- How close any level is to chance (baseline = majority-class rate)

Results are shown as a bar chart and a summary table.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier

# ── use T1 features only so modality can't trivially distinguish classes ──────
t1_mask = all_modalities == "T1"
y = all_labels[t1_mask]

# Build the feature sets to evaluate
feature_sets = {}
for i in range(nb_levels):
    feature_sets[f"Level {i} (all ch)"] = all_features[f"level_{i}"][t1_mask]

# Level-0 content / style subsets (only if content projection was active)
if all_content_indices is not None and len(all_style_indices) > 0:
    feature_sets["Level 0 — content"] = all_full_level0[t1_mask][:, all_content_indices]
    feature_sets["Level 0 — style"] = all_full_level0[t1_mask][:, all_style_indices]


# ── classifiers ───────────────────────────────────────────────────────────────
def make_lr():
    """Logistic Regression with standard scaling."""
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
        ]
    )


def make_rf():
    """Random Forest (no scaling needed)."""
    return RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
chance = cross_val_score(DummyClassifier(strategy="most_frequent"), np.zeros((len(y), 1)), y, cv=cv).mean()

print(f"{'Feature set':<28} {'LR acc':>8} {'LR ±':>7}  {'RF acc':>8} {'RF ±':>7}")
print("-" * 65)

results = []
for name, X in feature_sets.items():
    lr_scores = cross_val_score(make_lr(), X, y, cv=cv, scoring="accuracy")
    rf_scores = cross_val_score(make_rf(), X, y, cv=cv, scoring="accuracy")
    results.append(
        {
            "Feature set": name,
            "LR mean": lr_scores.mean(),
            "LR std": lr_scores.std(),
            "RF mean": rf_scores.mean(),
            "RF std": rf_scores.std(),
        }
    )
    print(
        f"{name:<28} {lr_scores.mean():.3f}   ±{lr_scores.std():.3f}   "
        f"{rf_scores.mean():.3f}   ±{rf_scores.std():.3f}"
    )

print("-" * 65)
print(f"{'Majority-class baseline':<28} {chance:.3f}")

results_df = pd.DataFrame(results)
print("\nSummary dataframe saved as  results_df  (use .to_csv() to export)")

In [ ]:
# ── Bar chart: LR and RF accuracy per feature set ────────────────────────────
x = np.arange(len(results_df))
width = 0.35

fig, ax = plt.subplots(figsize=(max(8, len(results_df) * 1.6), 5))

bars_lr = ax.bar(
    x - width / 2,
    results_df["LR mean"],
    width,
    yerr=results_df["LR std"],
    capsize=4,
    label="Logistic Regression",
    color="steelblue",
    alpha=0.85,
)

bars_rf = ax.bar(
    x + width / 2,
    results_df["RF mean"],
    width,
    yerr=results_df["RF std"],
    capsize=4,
    label="Random Forest",
    color="tomato",
    alpha=0.85,
)

# Chance-level line
ax.axhline(chance, color="grey", linestyle="--", linewidth=1.2, label=f"Chance ({chance:.2f})")

# Annotate bars with their mean accuracy
for bar in list(bars_lr) + list(bars_rf):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005, f"{h:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(results_df["Feature set"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("5-fold CV accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Diagnostic label prediction accuracy per VQ-VAE-2 feature set\n(T1 features only, 5-fold stratified CV)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("diagnostic_prediction_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved diagnostic_prediction_accuracy.png")

# ── Per-class breakdown (confusion matrix for best feature set) ──────────────
best_name = results_df.loc[results_df["RF mean"].idxmax(), "Feature set"]
best_X = feature_sets[best_name]
print(f"\nBest feature set by RF accuracy: '{best_name}'")
print("Showing confusion matrix (RF, 5-fold average) ...")

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_pred = cross_val_predict(make_rf(), best_X, y, cv=cv)
cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[label_names[i] for i in sorted(label_names)])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion matrix — RF on '{best_name}'")
plt.tight_layout()
plt.savefig("confusion_matrix_best_level.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved confusion_matrix_best_level.png")

print("\nClassification report:")
print(classification_report(y, y_pred, target_names=[label_names[i] for i in sorted(label_names)]))

In [ ]:
N_SHOW = 3  # number of subjects to show

fig, axes = plt.subplots(N_SHOW, 4, figsize=(14, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]

for row, item in enumerate(items[:N_SHOW]):
    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for col_offset, (key, mod_label) in enumerate([("image_t1", "T1"), ("image_t2", "T2")]):
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            recon, _, _, _, _, _ = vqvae_model(img, return_recon=True, pool_only=False)

        # Mid-sagittal slice (dim 2 = D)
        mid = img.shape[2] // 2
        orig_slice = img[0, 0, mid].cpu().numpy()
        recon_slice = recon[0, 0, mid].cpu().float().numpy()

        axes[row, col_offset * 2].imshow(orig_slice, cmap="gray", origin="lower")
        axes[row, col_offset * 2].set_title(f"Subject {row} — {mod_label} original")
        axes[row, col_offset * 2].axis("off")

        axes[row, col_offset * 2 + 1].imshow(recon_slice, cmap="gray", origin="lower")
        axes[row, col_offset * 2 + 1].set_title(f"Subject {row} — {mod_label} reconstruction")
        axes[row, col_offset * 2 + 1].axis("off")

plt.suptitle("VQ-VAE-2 Reconstructions (mid-sagittal)", fontsize=13)
plt.tight_layout()
plt.savefig("reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved reconstructions.png")

## 10. Save extracted features

In [ ]:
save_dict = {
    "labels": all_labels,
    "modalities": all_modalities,
    "label_map_keys": np.array(list(label_map.keys())),
    "label_map_values": np.array(list(label_map.values())),
    "checkpoint_step": np.array([checkpoint["step"]]),
    "content_indices": np.array(all_content_indices) if all_content_indices else np.array([]),
    "full_level0": all_full_level0,
}
for i in range(nb_levels):
    save_dict[f"features_level_{i}"] = all_features[f"level_{i}"]

np.savez("extracted_latents.npz", **save_dict)
print("✓ Saved extracted_latents.npz")
for i in range(nb_levels):
    print(f"  level_{i}: {all_features[f'level_{i}'].shape}")
print(f"  full_level0: {all_full_level0.shape}")

## 11. Codebook Analysis — Extract Indices

Run a separate forward pass with `return_recon=True` so that non-coarsest codebook
levels get proper decoder conditioning (without it they receive zero-padded input
and produce incorrect indices).

We collect:
- **Per-level codebook usage histograms** (normalized frequency of each entry per subject)
- **Raw codebook indices** for the code-replacement experiment later

In [ ]:
nb_entries = vqvae_module.codebooks[0].n_embed
print(f"Codebook: {nb_levels} levels, {nb_entries} entries each\n")

# Storage: index 0 = finest (level 0), index nb_levels-1 = coarsest
cb_histograms = [[] for _ in range(nb_levels)]  # per-subject normalized histograms
cb_labels = []  # diagnosis label per sample
cb_modalities = []  # "T1" or "T2"

# Per-class examples for code-replacement comparison (CN vs AD)
class_examples = {}

# Position-wise code frequency accumulators (T1 only).
# pos_counts[level] has shape (D_l, H_l, W_l, nb_entries) — lazily initialized.
pos_counts = [None for _ in range(nb_levels)]
n_t1_subjects = 0  # number of T1 subjects contributing to pos_counts

print(f"Extracting codebook indices from {len(items)} subjects (T1 + T2) ...")
print(f"Will keep one T1 example per class for code-replacement demo: {list(label_names.values())}")

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            recon, diffs, _, _, _, id_outputs_raw = vqvae_model(
                img,
                return_recon=True,
                pool_only=False,
            )

        # id_outputs_raw is coarsest-first; reverse to finest-first
        id_outputs = id_outputs_raw[::-1]

        for lvl in range(nb_levels):
            hist = torch.bincount(id_outputs[lvl].reshape(-1), minlength=nb_entries)
            hist = hist.float() / hist.sum()
            cb_histograms[lvl].append(hist.cpu().numpy())

        cb_labels.append(item["label"])
        cb_modalities.append(modality)

        # Accumulate position-wise code counts (T1 only to avoid modality confound)
        if modality == "T1":
            n_t1_subjects += 1
            for lvl in range(nb_levels):
                ids_lvl = id_outputs[lvl].squeeze(0).cpu()  # (D_l, H_l, W_l)
                if pos_counts[lvl] is None:
                    spatial = ids_lvl.shape
                    pos_counts[lvl] = torch.zeros(*spatial, nb_entries, dtype=torch.long)
                # Scatter-add: for each position, increment the count for the observed code
                pos_counts[lvl].scatter_(-1, ids_lvl.unsqueeze(-1), 1)

        # Keep one T1 example per diagnosis class
        if modality == "T1":
            class_name = label_names[item["label"]]
            if class_name not in class_examples:
                t1_tensor = transformed[key]
                ex_affine = None
                if hasattr(t1_tensor, "affine"):
                    ex_affine = t1_tensor.affine.numpy()
                elif hasattr(t1_tensor, "meta") and "affine" in t1_tensor.meta:
                    ex_affine = t1_tensor.meta["affine"].numpy()

                class_examples[class_name] = {
                    "id_outputs": [id_outputs[lvl].clone() for lvl in range(nb_levels)],
                    "image": img.clone(),
                    "item": item,
                    "affine": ex_affine,
                }
                print(f"  ✓ Stored {class_name} example: subject {item.get('subject', idx)}")

# Stack
for lvl in range(nb_levels):
    cb_histograms[lvl] = np.array(cb_histograms[lvl])
cb_labels = np.array(cb_labels)
cb_modalities = np.array(cb_modalities)

# Backward compat
_fallback_class = list(class_examples.keys())[-1] if class_examples else None
last_id_outputs = class_examples[_fallback_class]["id_outputs"] if _fallback_class else None
last_image = class_examples[_fallback_class]["image"] if _fallback_class else None
last_item = class_examples[_fallback_class]["item"] if _fallback_class else None
last_affine = class_examples[_fallback_class]["affine"] if _fallback_class else None

print(f"\n✓ Codebook extraction complete ({cb_histograms[0].shape[0]} samples)")
print(f"  T1 subjects for position-wise analysis: {n_t1_subjects}")
for lvl in range(nb_levels):
    used = (cb_histograms[lvl].sum(axis=0) > 0).sum()
    sp = tuple(pos_counts[lvl].shape[:-1]) if pos_counts[lvl] is not None else "?"
    print(
        f"  Level {lvl}: histogram shape {cb_histograms[lvl].shape}, "
        f"codes used: {used}/{nb_entries}, code map spatial: {sp}"
    )
print(f"  Class examples stored: {list(class_examples.keys())}")

## 12. Codebook Usage by Diagnosis Class

Mean codebook entry frequency per class, shown as overlaid histograms and a heatmap.
Entries that are heavily used by one class but not others indicate class-specific
structural patterns captured by that code vector.

In [ ]:
import seaborn as sns

# Use T1 only so modality doesn't confound the class comparison
t1_cb_mask = cb_modalities == "T1"
t1_cb_labels = cb_labels[t1_cb_mask]

for lvl in range(nb_levels):
    hists = cb_histograms[lvl][t1_cb_mask]

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f"Level {lvl} — Codebook Usage by Diagnosis (T1 only)", fontsize=14)

    # Mean usage histogram per class
    ax = axes[0]
    for lab_idx, lab_name in label_names.items():
        mask = t1_cb_labels == lab_idx
        mean_usage = hists[mask].mean(axis=0)
        ax.bar(np.arange(nb_entries), mean_usage, alpha=0.5, label=lab_name)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mean frequency")
    ax.set_title("Mean codebook usage")
    ax.legend()

    # Heatmap: classes x entries
    ax = axes[1]
    usage_matrix = np.zeros((len(label_names), nb_entries))
    for lab_idx in label_names:
        usage_matrix[lab_idx] = hists[t1_cb_labels == lab_idx].mean(axis=0)
    sns.heatmap(
        usage_matrix,
        ax=ax,
        cmap="viridis",
        yticklabels=[label_names[i] for i in sorted(label_names)],
        xticklabels=False,
    )
    ax.set_xlabel("Codebook entry")
    ax.set_title("Usage heatmap (class x entry)")

    plt.tight_layout()
    plt.savefig(f"codebook_usage_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Print top codes with biggest cross-class difference
    max_diff = usage_matrix.max(axis=0) - usage_matrix.min(axis=0)
    dominant_class = np.argmax(usage_matrix, axis=0)
    top_diff = np.argsort(max_diff)[::-1][:10]
    print(f"  Level {lvl} — codes with largest cross-class difference:")
    print(f"  {'Code':>6s}  {'Dominant':>8s}  {'MaxFreq':>8s}  {'MinFreq':>8s}  {'Diff':>8s}")
    for idx in top_diff:
        dom = label_names[dominant_class[idx]]
        print(
            f"  {idx:6d}  {dom:>8s}  {usage_matrix[:, idx].max():8.4f}  "
            f"{usage_matrix[:, idx].min():8.4f}  {max_diff[idx]:8.4f}"
        )
    print()

## 12b. Dominant Code Pairs by Spatial Position

For each spatial position in the code map, we look at which codebook entries appear
most often across all T1 subjects. If the **top-2 codes** at a position both exceed
a frequency threshold (e.g., 30% of subjects), that position exhibits a **dominant
pair** — meaning the model consistently alternates between two codes at that location.

We then aggregate these pairs across all positions and levels to find the most
common dominant pairs in the dataset. This reveals:
- Whether the codebook is being used efficiently (many distinct pairs) or
  degenerately (a few codes dominate everywhere)
- Spatial structure in code assignment (e.g., do certain brain regions always
  alternate between the same two codes?)

In [ ]:
from collections import Counter

# ── Configuration ────────────────────────────────────────────────
FREQ_THRESHOLD = 0.30  # both top-2 codes must appear in ≥ this fraction of subjects
TOP_K_PAIRS = 20  # how many top pairs to show per level

print(
    f"Frequency threshold: both codes must appear in ≥ {FREQ_THRESHOLD:.0%} of "
    f"{n_t1_subjects} T1 subjects (≥ {int(FREQ_THRESHOLD * n_t1_subjects)} subjects)\n"
)

all_level_results = {}

for lvl in range(nb_levels):
    counts = pos_counts[lvl]  # (D, H, W, nb_entries), long
    spatial_shape = counts.shape[:-1]
    n_positions = counts[..., 0].numel()

    # Convert to frequencies
    freqs = counts.float() / n_t1_subjects  # (D, H, W, nb_entries)

    # Top-2 codes and their frequencies at each position
    top2_freqs, top2_codes = freqs.topk(2, dim=-1)  # each (D, H, W, 2)
    freq1 = top2_freqs[..., 0]  # most common
    freq2 = top2_freqs[..., 1]  # second most common
    code1 = top2_codes[..., 0]
    code2 = top2_codes[..., 1]

    # Positions where both top-2 codes exceed threshold
    dominant_mask = (freq1 >= FREQ_THRESHOLD) & (freq2 >= FREQ_THRESHOLD)
    n_dominant = dominant_mask.sum().item()

    print(f"{'='*65}")
    print(f"Level {lvl}  —  spatial {tuple(spatial_shape)}  " f"({n_positions} positions)")
    print(
        f"  Positions with dominant pair (both ≥ {FREQ_THRESHOLD:.0%}): "
        f"{n_dominant} / {n_positions} ({n_dominant/n_positions:.1%})"
    )

    if n_dominant == 0:
        print("  No dominant pairs found at this level. " "Try lowering FREQ_THRESHOLD.\n")
        all_level_results[lvl] = {"n_dominant": 0, "pair_counts": Counter()}
        continue

    # Extract the dominant pairs as sorted tuples (so (a,b) == (b,a))
    c1 = code1[dominant_mask].numpy()
    c2 = code2[dominant_mask].numpy()
    f1 = freq1[dominant_mask].numpy()
    f2 = freq2[dominant_mask].numpy()

    pair_counter = Counter()
    pair_freq_sum = {}  # accumulate frequencies for averaging
    for i in range(len(c1)):
        pair = tuple(sorted([int(c1[i]), int(c2[i])]))
        pair_counter[pair] += 1
        if pair not in pair_freq_sum:
            pair_freq_sum[pair] = [0.0, 0.0, 0]  # sum_freq1, sum_freq2, count
        pair_freq_sum[pair][0] += f1[i]
        pair_freq_sum[pair][1] += f2[i]
        pair_freq_sum[pair][2] += 1

    all_level_results[lvl] = {
        "n_dominant": n_dominant,
        "pair_counts": pair_counter,
        "dominant_mask": dominant_mask,
    }

    # Print top pairs
    print(f"\n  Top-{min(TOP_K_PAIRS, len(pair_counter))} dominant pairs:")
    print(f"  {'Pair':<16} {'# positions':>12} {'% of dominant':>14} " f"{'Avg freq code1':>15} {'Avg freq code2':>15}")
    print(f"  {'-'*74}")

    for pair, count in pair_counter.most_common(TOP_K_PAIRS):
        pct = count / n_dominant * 100
        s = pair_freq_sum[pair]
        avg_f1 = s[0] / s[2]
        avg_f2 = s[1] / s[2]
        print(
            f"  ({pair[0]:>3d}, {pair[1]:>3d})     {count:>8d}     {pct:>10.1f}%"
            f"     {avg_f1:>11.1%}     {avg_f2:>11.1%}"
        )

    # Summary stats
    n_unique_pairs = len(pair_counter)
    n_unique_codes = len(set(c for pair in pair_counter for c in pair))
    print(f"\n  Unique dominant pairs: {n_unique_pairs}")
    print(f"  Unique codes involved: {n_unique_codes} / {nb_entries}")

    # Distribution of how many positions each pair covers
    pair_sizes = list(pair_counter.values())
    print(
        f"  Positions per pair: min={min(pair_sizes)}, "
        f"median={sorted(pair_sizes)[len(pair_sizes)//2]}, "
        f"max={max(pair_sizes)}"
    )
    print()

# ── Cross-level summary ──────────────────────────────────────────
print("=" * 65)
print("Cross-level summary")
print("=" * 65)

# Find pairs that appear as dominant at multiple levels
all_pairs = Counter()
for lvl in range(nb_levels):
    for pair in all_level_results[lvl]["pair_counts"]:
        all_pairs[pair] += all_level_results[lvl]["pair_counts"][pair]

# Pairs that are dominant at more than one level
multi_level_pairs = []
for pair in all_pairs:
    levels_present = [lvl for lvl in range(nb_levels) if pair in all_level_results[lvl]["pair_counts"]]
    if len(levels_present) > 1:
        total = sum(all_level_results[l]["pair_counts"][pair] for l in levels_present)
        multi_level_pairs.append((pair, levels_present, total))

multi_level_pairs.sort(key=lambda x: -x[2])

if multi_level_pairs:
    print(f"\nPairs dominant at multiple levels ({len(multi_level_pairs)} pairs):")
    for pair, levels, total in multi_level_pairs[:15]:
        lvl_str = ", ".join(str(l) for l in levels)
        print(f"  ({pair[0]:>3d}, {pair[1]:>3d})  levels=[{lvl_str}]  " f"total positions={total}")
else:
    print("\nNo pairs are dominant at multiple levels.")

# Overall code utilization in dominant positions
all_dominant_codes = set()
for lvl in range(nb_levels):
    for pair in all_level_results[lvl]["pair_counts"]:
        all_dominant_codes.update(pair)
print(
    f"\nTotal unique codes in dominant pairs (all levels): "
    f"{len(all_dominant_codes)} / {nb_entries} "
    f"({len(all_dominant_codes)/nb_entries:.1%})"
)

## 13. Most Discriminative Codes — Mutual Information & Chi-squared

**Mutual information** measures how much knowing a codebook entry's usage reduces
uncertainty about the diagnosis label (higher = more informative).

**Chi-squared** tests whether the usage distribution of each entry is independent
of the class label (higher statistic = stronger association).

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import chi2_contingency

for lvl in range(nb_levels):
    hists = cb_histograms[lvl][t1_cb_mask]
    y = t1_cb_labels

    # ── Mutual Information ────────────────────────────────────────────
    mi_scores = mutual_info_classif(hists, y, discrete_features=False, random_state=42)

    fig, axes = plt.subplots(1, 2, figsize=(18, 4))
    fig.suptitle(f"Level {lvl} — Codebook Discriminativeness (T1 only)", fontsize=14)

    ax = axes[0]
    ax.bar(np.arange(nb_entries), mi_scores, color="darkorange", width=1.0)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mutual information (nats)")
    ax.set_title("MI: codebook entry usage vs diagnosis")

    top_mi = np.argsort(mi_scores)[::-1][:10]
    print(f"Level {lvl} — top-10 MI codes: {top_mi.tolist()}")
    print(f"  MI values: {mi_scores[top_mi].round(4).tolist()}")

    # ── Chi-squared ───────────────────────────────────────────────────
    chi2_scores = np.zeros(nb_entries)
    for code_idx in range(nb_entries):
        usage = hists[:, code_idx]
        # Bin non-zero usage into terciles for the contingency table
        nonzero = usage[usage > 0]
        if len(nonzero) < 10:
            continue
        bins = np.quantile(nonzero, [0.33, 0.66])
        if len(np.unique(bins)) < 2:
            continue
        digitized = np.digitize(usage, bins)
        contingency = pd.crosstab(digitized, y)
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            chi2, p, _, _ = chi2_contingency(contingency)
            chi2_scores[code_idx] = chi2

    TOP_K = 20
    top_chi2 = np.argsort(chi2_scores)[::-1][:TOP_K]

    ax = axes[1]
    ax.bar(range(TOP_K), chi2_scores[top_chi2], color="steelblue")
    ax.set_xticks(range(TOP_K))
    ax.set_xticklabels(top_chi2, rotation=45)
    ax.set_xlabel("Codebook entry index")
    ax.set_ylabel("Chi-squared statistic")
    ax.set_title(f"Top {TOP_K} most class-discriminative codes (chi2)")

    plt.tight_layout()
    plt.savefig(f"codebook_discriminativeness_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print()

## 14. PCA & t-SNE of Codebook Usage Histograms

Each subject is represented by a vector of codebook entry frequencies. If the
codebook has learned class-relevant structure, subjects with the same diagnosis
should cluster together in this histogram space.

In [ ]:
for lvl in range(nb_levels):
    hists = cb_histograms[lvl][t1_cb_mask]

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(hists)
    ev = pca.explained_variance_ratio_

    tsne = TSNE(n_components=2, perplexity=min(30, len(hists) - 1), random_state=42)
    X_tsne = tsne.fit_transform(hists)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Level {lvl} — Codebook Histogram Embeddings (T1 only)", fontsize=14)

    for lab_idx, lab_name in label_names.items():
        m = t1_cb_labels == lab_idx
        axes[0].scatter(X_pca[m, 0], X_pca[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)
        axes[1].scatter(X_tsne[m, 0], X_tsne[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)

    axes[0].set_title(f"PCA ({ev.sum()*100:.1f}% var)")
    axes[0].set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    axes[0].set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")
    axes[0].legend(fontsize=8)

    axes[1].set_title("t-SNE")
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"codebook_histogram_embeddings_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()

# Combined: concatenate histograms across all levels
X_all = np.concatenate([cb_histograms[lvl][t1_cb_mask] for lvl in range(nb_levels)], axis=1)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_all)
ev = pca.explained_variance_ratio_
tsne = TSNE(n_components=2, perplexity=min(30, len(X_all) - 1), random_state=42)
X_tsne = tsne.fit_transform(X_all)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("All Levels Combined — Codebook Histogram Embeddings (T1 only)", fontsize=14)
for lab_idx, lab_name in label_names.items():
    m = t1_cb_labels == lab_idx
    axes[0].scatter(X_pca[m, 0], X_pca[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)
    axes[1].scatter(X_tsne[m, 0], X_tsne[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)
axes[0].set_title(f"PCA ({ev.sum()*100:.1f}% var)")
axes[0].legend(fontsize=8)
axes[1].set_title("t-SNE")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig("codebook_histogram_embeddings_all_levels.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved codebook histogram embedding plots")

## 15. Codebook Vector Replacement — CN vs AD Comparison

Replace a specific codebook entry at a chosen level with a different entry, decode
back to an image, and compare the effect on a **healthy (CN)** and **diseased (AD)**
subject side by side. This reveals whether different codes capture class-specific
structural information (e.g., atrophy patterns in AD).

The model's `decode_codes` method handles the hierarchical decoding and automatically
provides a zero-style placeholder when `inject_style_to_decoder` is active.

In [ ]:
import torch.nn.functional as F

# ── Configuration ────────────────────────────────────────────────
TARGET_LEVEL = nb_levels - 1  # coarsest level (most semantic); change to 0 for finest
SLICE_AXIS = 2  # 0=sagittal, 1=coronal, 2=axial
COMPARE_CLASSES = ["CN", "AD"]  # classes to compare side by side

# Verify both classes are available
for cls in COMPARE_CLASSES:
    assert cls in class_examples, (
        f"Class '{cls}' not found in class_examples. " f"Available: {list(class_examples.keys())}"
    )

# ── Auto-select codes to swap ───────────────────────────────────
# Use the CN example to pick codes (most-used → least-used)
cn_ids = class_examples["CN"]["id_outputs"][TARGET_LEVEL]
unique_codes, counts = cn_ids.unique(return_counts=True)
OLD_CODE = unique_codes[counts.argmax()].item()

NEW_CODE = unique_codes[counts.argmin()].item()
if NEW_CODE == OLD_CODE:
    all_present = set(unique_codes.cpu().tolist())
    absent = [c for c in range(nb_entries) if c not in all_present]
    NEW_CODE = absent[0] if absent else (OLD_CODE + 1) % nb_entries

print(f"Target level: {TARGET_LEVEL} (0=finest, {nb_levels-1}=coarsest)")
print(f"Replacing code {OLD_CODE} -> {NEW_CODE}")
print(f"Comparing classes: {COMPARE_CLASSES}\n")

# ── Embedding diagnostics ───────────────────────────────────────
codebook = vqvae_module.codebooks[TARGET_LEVEL]
emb_old = codebook.embed[:, OLD_CODE]
emb_new = codebook.embed[:, NEW_CODE]
l2_dist = (emb_old - emb_new).norm().item()
cos_sim = F.cosine_similarity(emb_old.unsqueeze(0), emb_new.unsqueeze(0)).item()
print(f"Embedding comparison (code {OLD_CODE} vs {NEW_CODE}):")
print(f"  L2 distance:       {l2_dist:.4f}")
print(f"  Cosine similarity: {cos_sim:.4f}")

all_embeds = codebook.embed.T
sample_idx = torch.randint(0, all_embeds.shape[0], (min(200, nb_entries),))
sample_embeds = all_embeds[sample_idx]
pairwise = torch.cdist(sample_embeds.unsqueeze(0), sample_embeds.unsqueeze(0)).squeeze()
tri_mask = torch.triu(torch.ones_like(pairwise, dtype=torch.bool), diagonal=1)
print(f"  Pairwise L2 stats (sample): mean={pairwise[tri_mask].mean():.4f}, " f"std={pairwise[tri_mask].std():.4f}\n")

# ── Run replacement for each class ──────────────────────────────
class_results = {}  # class_name -> dict with recon_orig, recon_modified, diff, n_replaced, image

for cls in COMPARE_CLASSES:
    ex = class_examples[cls]
    ids = ex["id_outputs"]
    img = ex["image"]
    subj = ex["item"].get("subject", "?")

    print(f"--- {cls} (subject {subj}) ---")
    print(f"  Code map shape at level {TARGET_LEVEL}: {tuple(ids[TARGET_LEVEL].shape)}")

    # Replace codes
    modified_ids = [i.clone() for i in ids]
    n_replaced = (modified_ids[TARGET_LEVEL] == OLD_CODE).sum().item()
    modified_ids[TARGET_LEVEL][modified_ids[TARGET_LEVEL] == OLD_CODE] = NEW_CODE
    print(f"  Replaced {n_replaced} voxels from code {OLD_CODE} -> {NEW_CODE}")

    with torch.no_grad():
        recon_orig = vqvae_module.decode_codes(*ids)
        recon_modified = vqvae_module.decode_codes(*modified_ids)

    # Interpolate to input spatial size if needed
    input_shape = img.shape[2:]
    if recon_orig.shape[2:] != input_shape:
        recon_orig = F.interpolate(recon_orig, size=input_shape, mode="trilinear", align_corners=False)
    if recon_modified.shape[2:] != input_shape:
        recon_modified = F.interpolate(recon_modified, size=input_shape, mode="trilinear", align_corners=False)

    diff = (recon_modified - recon_orig).abs()
    print(f"  Diff — max={diff.max().item():.6f}, mean={diff.mean().item():.6f}")
    rel_max = diff.max().item() / (recon_orig.abs().max().item() + 1e-8)
    print(f"  Relative max diff: {rel_max:.4%}")

    class_results[cls] = {
        "recon_orig": recon_orig,
        "recon_modified": recon_modified,
        "diff": diff,
        "n_replaced": n_replaced,
        "image": img,
        "subject": subj,
    }


# ── Visualize side by side ──────────────────────────────────────
def get_slices(vol, axis):
    """Get quarter, mid, and three-quarter slices from (C, D, H, W)."""
    n = vol.shape[axis + 1]
    return [
        vol[0].select(axis, n // 4).cpu().numpy(),
        vol[0].select(axis, n // 2).cpu().numpy(),
        vol[0].select(axis, 3 * n // 4).cpu().numpy(),
    ]


n_classes = len(COMPARE_CLASSES)
# Layout: rows = slice positions (Q1, Mid, Q3)
# For each class: Original | Recon | Modified | Diff  → 4 cols per class
n_cols = 4 * n_classes
slice_labels = ["Q1", "Mid", "Q3"]

# Compute shared intensity ranges across classes for consistent display
all_recons = [class_results[c]["recon_orig"] for c in COMPARE_CLASSES] + [
    class_results[c]["recon_modified"] for c in COMPARE_CLASSES
]
vmin_r = min(r.min().item() for r in all_recons)
vmax_r = max(r.max().item() for r in all_recons)
diff_max = max(max(class_results[c]["diff"].max().item(), 1e-8) for c in COMPARE_CLASSES)

fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 14))

for ci, cls in enumerate(COMPARE_CLASSES):
    r = class_results[cls]
    col_offset = ci * 4
    volumes = [r["image"], r["recon_orig"], r["recon_modified"], r["diff"]]
    titles = [
        f"{cls} (subj {r['subject']})\nOriginal input",
        f"{cls}\nDecoded (orig codes)",
        f"{cls}\nDecoded (modified)",
        f"{cls}\nDifference",
    ]

    for col_i, (vol, title) in enumerate(zip(volumes, titles)):
        slices = get_slices(vol[0], SLICE_AXIS)
        for row in range(3):
            ax = axes[row, col_offset + col_i]
            if "difference" in title.lower():
                im = ax.imshow(slices[row], cmap="hot", origin="lower", vmin=0, vmax=diff_max)
                if row == 0:
                    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            elif "input" in title.lower():
                ax.imshow(slices[row], cmap="gray", origin="lower")
            else:
                ax.imshow(slices[row], cmap="gray", origin="lower", vmin=vmin_r, vmax=vmax_r)
            if row == 0:
                ax.set_title(title, fontsize=10)
            ax.set_ylabel(slice_labels[row], fontsize=9) if col_i == 0 else None
            ax.axis("off")

fig.suptitle(
    f"Code Replacement — Level {TARGET_LEVEL}, code {OLD_CODE} → {NEW_CODE}\n"
    f"L2 dist={l2_dist:.4f}, cos_sim={cos_sim:.4f}, "
    f"shared diff scale max={diff_max:.6f}",
    fontsize=13,
)
plt.tight_layout()
plt.savefig("codebook_replacement_cn_vs_ad.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Summary table ───────────────────────────────────────────────
print("\nSummary:")
print(f"{'Class':<8} {'Subject':<12} {'# replaced':>12} {'Max diff':>12} {'Mean diff':>12} {'Rel max':>12}")
print("-" * 72)
for cls in COMPARE_CLASSES:
    r = class_results[cls]
    rel = r["diff"].max().item() / (r["recon_orig"].abs().max().item() + 1e-8)
    print(
        f"{cls:<8} {str(r['subject']):<12} {r['n_replaced']:>12d} "
        f"{r['diff'].max().item():>12.6f} {r['diff'].mean().item():>12.6f} {rel:>11.4%}"
    )

print("\n✓ Saved codebook_replacement_cn_vs_ad.png")

## 16. Save Reconstructions as NIfTI — CN vs AD

Save the original input, VQ-VAE reconstruction, code-modified reconstruction, and
absolute difference map as NIfTI files (`.nii.gz`) for **each class** (CN, AD).
The affine matrix is taken from MONAI's post-transform MetaTensor so the saved
volumes are in the correct preprocessed coordinate space (RAS orientation, resampled,
padded/cropped) and can be loaded directly in FSLeyes, ITK-SNAP, or 3D Slicer.

In [ ]:
import nibabel as nib

# ── Output directory ─────────────────────────────────────────────
NIFTI_OUT_DIR = "nifti_outputs"
os.makedirs(NIFTI_OUT_DIR, exist_ok=True)


# ── Helper: build fallback affine from preprocessing params ──────
def _make_fallback_affine():
    """RAS-aligned affine with the correct spacing, centred at origin."""
    D, H, W = spatial_size
    aff = np.eye(4)
    aff[0, 0] = spacing
    aff[1, 1] = spacing
    aff[2, 2] = spacing
    aff[0, 3] = -spacing * (D - 1) / 2.0
    aff[1, 3] = -spacing * (H - 1) / 2.0
    aff[2, 3] = -spacing * (W - 1) / 2.0
    return aff


# ── Helper: save tensor → .nii.gz ───────────────────────────────
def save_nifti(tensor, affine, filepath, description=""):
    """Save a torch tensor as a compressed NIfTI file.

    Args:
        tensor: (1, 1, D, H, W) or (1, D, H, W) or (D, H, W) torch tensor.
        affine: 4x4 numpy affine matrix.
        filepath: Output path (should end in .nii.gz).
        description: Optional description for the NIfTI header.
    """
    vol = tensor.detach().cpu().float().numpy()
    while vol.ndim > 3:
        vol = vol[0]
    img = nib.Nifti1Image(vol, affine)
    if description:
        img.header["descrip"] = description.encode("ascii", errors="ignore")[:80]
    nib.save(img, filepath)
    print(f"  Saved: {filepath}  {vol.shape}  [{vol.min():.4f}, {vol.max():.4f}]")


# ── Save volumes for each class ──────────────────────────────────
for cls in COMPARE_CLASSES:
    ex = class_examples[cls]
    r = class_results[cls]
    subject_id = ex["item"].get("subject", "unknown")

    # Use MetaTensor affine if available, else fallback
    affine = ex["affine"].copy() if ex["affine"] is not None else _make_fallback_affine()
    affine_src = "MetaTensor" if ex["affine"] is not None else "constructed"

    print(f"\n{'='*60}")
    print(f"{cls} — subject {subject_id}  (affine: {affine_src})")
    print(f"{'='*60}")

    prefix = f"{NIFTI_OUT_DIR}/{cls}_subj_{subject_id}"

    save_nifti(
        r["image"],
        affine,
        f"{prefix}_input.nii.gz",
        description=f"{cls} VQ-VAE input (preprocessed T1)",
    )

    save_nifti(
        r["recon_orig"],
        affine,
        f"{prefix}_recon_orig.nii.gz",
        description=f"{cls} VQ-VAE reconstruction (original codes)",
    )

    save_nifti(
        r["recon_modified"],
        affine,
        f"{prefix}_recon_modified_L{TARGET_LEVEL}_c{OLD_CODE}to{NEW_CODE}.nii.gz",
        description=f"{cls} recon (L{TARGET_LEVEL}: {OLD_CODE}->{NEW_CODE})",
    )

    save_nifti(
        r["diff"],
        affine,
        f"{prefix}_diff_L{TARGET_LEVEL}_c{OLD_CODE}to{NEW_CODE}.nii.gz",
        description=f"{cls} diff (L{TARGET_LEVEL}: {OLD_CODE}->{NEW_CODE})",
    )

print(f"\n✓ All NIfTI volumes saved to {NIFTI_OUT_DIR}/")
print("  Open in FSLeyes / ITK-SNAP / 3D Slicer for interactive inspection.")
print(f"  Affine matches preprocessed space (RAS, {spacing}mm, {spatial_size}).")
print(f"\n  Files per class:")
for cls in COMPARE_CLASSES:
    subj = class_examples[cls]["item"].get("subject", "unknown")
    print(f"    {cls} (subj {subj}): *input, *recon_orig, *recon_modified, *diff")